In [1]:
import sys
import os

src_path = os.path.abspath("../src")
sys.path.append(src_path)

from lingo_parser.parser import *
from lingo_parser.transformer import *


from pyomo_generator.json_parser import *
from notebook_generator.notebook_construct import *

from excel_parser.excel_module import *

In [29]:
tree = parse_lingo_model("../data/Regime_explicit.lng")
model_dict = LingoModelTransformer2().transform(tree)
pyomo_code = generate_pyomo_code(model_dict)
print(pyomo_code)

from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.ALIMENTS = Set(initialize=['Brownie', 'Cremeglacee', 'Cola', 'Gateau'])
model.INGREDIENTS = Set(initialize=['Calories', 'Chocolat', 'Sucre', 'Matieregrasse'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.ALIMENTS for j in model.INGREDIENTS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Prix = Param(model.ALIMENTS, initialize={'Brownie': 50.0, 'Cremeglacee': 20.0, 'Cola': 30.0, 'Gateau': 80.0}, within=NonNegativeReals)
model.Calories = Param(model.ALIMENTS, initialize={'Brownie': 400.0, 'Cremeglacee': 200.0, 'Cola': 150.0, 'Gateau': 500.0}, within=NonNegativeReals)
model.Dietejour = Param(model.INGREDIENTS, initial

In [30]:
from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.ALIMENTS = Set(initialize=['Brownie', 'Cremeglacee', 'Cola', 'Gateau'])
model.INGREDIENTS = Set(initialize=['Calories', 'Chocolat', 'Sucre', 'Matieregrasse'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.ALIMENTS for j in model.INGREDIENTS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Prix = Param(model.ALIMENTS, initialize={'Brownie': 50.0, 'Cremeglacee': 20.0, 'Cola': 30.0, 'Gateau': 80.0}, within=NonNegativeReals)
model.Calories = Param(model.ALIMENTS, initialize={'Brownie': 400.0, 'Cremeglacee': 200.0, 'Cola': 150.0, 'Gateau': 500.0}, within=NonNegativeReals)
model.Dietejour = Param(model.INGREDIENTS, initialize={'Calories': 500.0, 'Chocolat': 6.0, 'Sucre': 10.0, 'Matieregrasse': 8.0}, within=NonNegativeReals)
model.Qteing = Param(model.ALIMENTS, model.INGREDIENTS, initialize={('Brownie', 'Calories'): 400.0, ('Brownie', 'Chocolat'): 3.0, ('Brownie', 'Sucre'): 2.0, ('Brownie', 'Matieregrasse'): 2.0, ('Cremeglacee', 'Calories'): 200.0, ('Cremeglacee', 'Chocolat'): 2.0, ('Cremeglacee', 'Sucre'): 2.0, ('Cremeglacee', 'Matieregrasse'): 4.0, ('Cola', 'Calories'): 150.0, ('Cola', 'Chocolat'): 0.0, ('Cola', 'Sucre'): 4.0, ('Cola', 'Matieregrasse'): 1.0, ('Gateau', 'Calories'): 500.0, ('Gateau', 'Chocolat'): 0.0, ('Gateau', 'Sucre'): 4.0, ('Gateau', 'Matieregrasse'): 5.0}, within=NonNegativeReals)

#==============================================================================
# VARIABLES
#==============================================================================

model.X = Var(model.ALIMENTS, domain=NonNegativeReals)

#==============================================================================
# CONSTRAINTS
#==============================================================================

model.c0 = Constraint(expr=sum(model.Calories[a]*model.X[a] for a in model.ALIMENTS) >= 500)
model.c_for_0 = ConstraintList()
for i in model.INGREDIENTS:
    model.c_for_0.add(sum(model.Qteing[a,i] * model.X[a] for a in model.ALIMENTS) >= model.Dietejour[i])

#==============================================================================
# OBJECTIVE
#==============================================================================

model.obj = Objective(expr=sum(model.Prix[a] * model.X[a] for a in model.ALIMENTS), sense=minimize)

In [31]:


# === Résolution ===
solver = SolverFactory('highs')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)


print("Valeur objectif : ",value(model.obj))
for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v}')
    for index in v:
        print(f'   {index} = {v[index].value}')

Valeur objectif :  90.0
Variable set: X
   Brownie = 0.0
   Cremeglacee = 3.0
   Cola = 1.0
   Gateau = 0.0


In [5]:


def generate_notebook() :

    tree = parse_lingo_model("../data/fleuriste_alg.lng")
    model_dict = LingoModelTransformer2().transform(tree)

    pyomo_code = generate_pyomo_code(model_dict)

    generate_pyomo_notebook(pyomo_code, solver="gurobi", filename="fleuristeaaa.ipynb")

generate_notebook()

✅ Notebook généré : fleuristeaaa.ipynb


In [2]:
convert_lingo_ole_to_explicit("../data/Cargo.lng")

'/Users/joaquim/Documents/CODE/Lingpy/lingo_to_pyomo/data/Cargo_explicit.lng'